# 05 · MobileNet / EfficientNet on CIFAR-10 —— 效率时代：同一份算力能买多少精度

**家族位置**：`02_CNN_Family` 第 5 个项目（效率时代收口）。

**与上一站的衔接**：04 末尾留下效率视角的悬念——DenseNet "参数省、计算贵"。本章给家族补上第二把尺子 **FLOPs**，并引入两个把"省"做到极致的架构：**MobileNet**（深度可分离分解）与 **EfficientNet**（复合缩放，其 MBConv 内置 SE——04 的选学伏笔在此转正）。

**学习目标**
1. 深度可分离分解的参数/FLOPs 账（现场算 7.9× 的账）
2. width_mult（α）宽度旋钮的收益-成本曲线
3. 复合缩放 vs 单轴缩放（mini 三臂对照）
4. Acc-FLOPs 散点图：把家族五个架构放到同一张"性价比地图"上

## 1. 原理：把卷积"拆开卖"与"按预算放大"

### 通俗理解

**一句话**：MobileNet 把一个"看所有通道再输出"的普通卷积拆成两步——先用每通道一个独立小卷积各自提炼（DW），再用 1×1 把各通道的结论混合（PW）；EfficientNet 则回答"预算变多时，该把钱花在加宽、加深还是加分辨率"——答案是按固定比例同时放大。

**比喻**：拆卷积像餐厅改革——原来一个全能厨师（普通卷积）同时看 32 个锅炒 64 道菜；改革后 32 个学徒各看各的锅（DW），再由一个传菜员统一调配（PW）。人力省近 8 倍，这就是深度可分离。

### 分解账（本项目现场验证）

标准 3×3 卷积：MACs = 9·Cin·Cout；分解后：DW 9·Cin + PW Cin·Cout。Cin=32, Cout=64, 32×32：**18.87M → 2.39M（7.9×）**。

### 复合缩放

EfficientNet 的观察：单轴放大边际递减；用系数 φ 同步放大 depth(α^φ)/width(β^φ)/resolution(γ^φ)，约束 α·β²·γ²≈2 保证每档 FLOPs 翻倍。CIFAR 32×32 无分辨率下探空间，本项目 mini 版演示 width/depth 两轴。

### FLOPs 口径

`count_flops` 统计 Conv/Linear 的 MACs（乘加次数），BN/激活/池化忽略；MACs×2≈FLOPs，模型间相对比较不受口径影响。

In [ ]:
import sys
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import CIFAR10_CLASSES, load_cifar10_torch
from common.engine import fit
from common.models import DenseNetCIFAR, EfficientNetCIFAR, InceptionCIFAR, MobileNetCIFAR, ResNetCIFAR
from common.utils import count_flops, count_params, set_seed, setup_chinese_font

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("设备:", DEVICE, "| torch:", torch.__version__)

## 2. 数据：CIFAR-10（同 03/04）

同一份数据、同一子集划分（前 10k）、同一标准化——跨项目锚点严格可比。

In [ ]:
Xtr, ytr, Xte, yte = load_cifar10_torch(str(ROOT / "data"))
print("训练集:", Xtr.shape, "| 测试集:", Xte.shape)

fig, axes = plt.subplots(2, 10, figsize=(12, 3.0))
for r in range(2):
    for c in range(10):
        idx = int(np.where(ytr.numpy() == c)[0][r])
        img = Xtr[idx].permute(1, 2, 0).numpy()
        img = (img * [0.2470, 0.2435, 0.2616] + [0.4914, 0.4822, 0.4465]).clip(0, 1)
        axes[r, c].imshow(img)
        axes[r, c].set_title(CIFAR10_CLASSES[c], fontsize=8)
        axes[r, c].axis("off")
plt.suptitle("CIFAR-10：每类两个样本", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig0_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. 模型定稿与两笔账

先算**分解账**（DW 为什么省），再核对六个配置的参数/MACs 全家福。

In [ ]:
# 分解账：同规格（32→64, 3×3, 32×32）标准卷积 vs DW+PW
std = nn.Conv2d(32, 64, 3, padding=1, bias=False)


class DWPW(nn.Module):
    def __init__(self):
        super().__init__()
        self.dw = nn.Conv2d(32, 32, 3, padding=1, groups=32, bias=False)
        self.pw = nn.Conv2d(32, 64, 1, bias=False)

    def forward(self, x):
        return self.pw(self.dw(x))


f_std = count_flops(std, (1, 32, 32, 32))
f_dwpw = count_flops(DWPW(), (1, 32, 32, 32))
print(f"分解账（32→64, 3×3, 32×32）: std={f_std/1e6:.2f}M vs DW+PW={f_dwpw/1e6:.2f}M MACs → 省 {f_std / f_dwpw:.1f}×")

CONFIGS = {
    "ResNet20":        lambda: ResNetCIFAR(),
    "DenseNetCIFAR":   lambda: DenseNetCIFAR(),
    "InceptionCIFAR":  lambda: InceptionCIFAR(),
    "MobileNet α=1":   lambda: MobileNetCIFAR(),
    "MobileNet α=0.5": lambda: MobileNetCIFAR(alpha=0.5),
    "EffNet base":     lambda: EfficientNetCIFAR(),
}
for name, cls in CONFIGS.items():
    m = cls()
    print(f"{name:16s} 参数量={count_params(m):>8,} | MACs={count_flops(m)/1e6:6.1f}M")

## 4. 主实验：效率两兄弟 + α 消融（约 10 分钟 CPU）

**协议（沿用 03/04 最终配方）**：10k 训练子集、10 epochs、SGD(momentum=0.9, lr=0.05) + weight_decay=1e-4、batch 128、seed=0、测试全量 10k、无增强。锚点直接引用 04 同协议实测（ResNet20 57.14% / DenseNet 58.77% / Inception 39.08%），不重训。

In [ ]:
EPOCHS = 10
tr = DataLoader(TensorDataset(Xtr[:10000], ytr[:10000]), batch_size=128, shuffle=True)
te = DataLoader(TensorDataset(Xte, yte), batch_size=512)

results = {}
for name, cls in [("MobileNet α=1", MobileNetCIFAR),
                  ("MobileNet α=0.5", lambda: MobileNetCIFAR(alpha=0.5)),
                  ("EffNet base", EfficientNetCIFAR)]:
    set_seed(0)
    model = cls()
    hist = fit(model, tr, te, epochs=EPOCHS, lr=0.05, device=DEVICE,
               optimizer_cls=partial(torch.optim.SGD, momentum=0.9),
               weight_decay=1e-4, verbose=False)
    results[name] = {"hist": hist, "model": model}
    print(f"{name:16s} val_acc={hist['val_acc'][-1]:.2%} | val_loss={hist['val_loss'][-1]:.4f} | train_acc={hist['train_acc'][-1]:.2%}", flush=True)

print("\n锚点（04 同协议实测）: ResNet20=57.14% | DenseNet=58.77% | Inception(轻量)=39.08%")

In [ ]:
colors = {"MobileNet α=1": "#4C72B0", "MobileNet α=0.5": "#937860", "EffNet base": "#55A868",
          "ResNet20": "#DD8452", "DenseNetCIFAR": "#8172B2", "InceptionCIFAR": "#C44E52"}

fig, ax = plt.subplots(figsize=(8.5, 4.2))
for name, c in colors.items():
    if name in results:
        ax.plot(results[name]["hist"]["val_acc"], marker="o", ms=4, label=name, color=c)
    else:
        ref = {"ResNet20": 0.5714, "DenseNetCIFAR": 0.5877, "InceptionCIFAR": 0.3908}
        ax.axhline(ref[name], color=c, ls="--", lw=1.2, alpha=0.8, label=f"{name}（锚点@04）")
ax.set_xlabel("epoch"); ax.set_ylabel("val_acc")
ax.set_title("效率时代 vs 家族锚点（10k 子集 · 10 epochs · SGD-momentum-wd · seed=0）")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGS / "fig1_curves.png", dpi=150, bbox_inches="tight")
plt.show()

accs = {"ResNet20": 0.5714, "DenseNetCIFAR": 0.5877, "InceptionCIFAR": 0.3908}
accs.update({n: results[n]["hist"]["val_acc"][-1] for n in results})

params = {n: count_params(cls()) for n, cls in CONFIGS.items()}
macs = {n: count_flops(cls()) for n, cls in CONFIGS.items()}

## 5. 招牌图：Acc-FLOPs 性价比地图

本站的核心交付：家族六个配置放上同一张 "精度-COMPUTE" 平面。注意这里特意**两张图并排**——Acc-vs-MACs 和 Acc-vs-params，因为 04 已经证明两者会给出不同答案（DenseNet 参数省、计算贵）。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for name, c in colors.items():
    axes[0].scatter(macs[name] / 1e6, accs[name], s=170, color=c, zorder=3)
    axes[0].annotate(f"{name}\n{accs[name]:.1%} · {macs[name]/1e6:.0f}M",
                     (macs[name] / 1e6, accs[name]), textcoords="offset points",
                     xytext=(8, -16), fontsize=8)
    axes[1].scatter(params[name] / 1e3, accs[name], s=170, color=c, zorder=3)
    axes[1].annotate(f"{name}\n{accs[name]:.1%} · {params[name]/1e3:.0f}k",
                     (params[name] / 1e3, accs[name]), textcoords="offset points",
                     xytext=(8, -16), fontsize=8)
axes[0].set_xlabel("MACs (百万/图)"); axes[0].set_ylabel("val_acc")
axes[0].set_title("精度 vs 计算量（Acc-FLOPs 地图）"); axes[0].grid(alpha=0.3)
axes[1].set_xlabel("参数量 (千)"); axes[1].set_ylabel("val_acc")
axes[1].set_title("精度 vs 参数量"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS / "fig2_acc_flops.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"{'模型':16s} {'参数量':>9s} {'MACs':>8s} {'val_acc':>8s}")
for n in CONFIGS:
    print(f"{n:16s} {params[n]:>9,} {macs[n]/1e6:>7.1f}M {accs[n]:>8.2%}")

## 6. 复合缩放三臂对照（约 20 分钟 CPU）

EfficientNet 的招牌主张：**按比例协同放大优于单轴猛冲**。三臂（同 10 epochs 预算）：base / 宽×2 / 宽×2+深×2。观察：加大后精度涨多少、计算涨多少、以及复合臂是否比单轴臂"每单位算力买得更多"。

In [ ]:
import time
SCALES = {
    "base":     dict(),
    "宽×2":     dict(width_mult=2.0),
    "宽×2+深×2": dict(width_mult=2.0, depth_mult=2.0),
}
scale_res = {}
for name, kw in SCALES.items():
    set_seed(0)
    model = EfficientNetCIFAR(**kw)
    t0 = time.time()
    hist = fit(model, tr, te, epochs=EPOCHS, lr=0.05, device=DEVICE,
               optimizer_cls=partial(torch.optim.SGD, momentum=0.9),
               weight_decay=1e-4, verbose=False)
    scale_res[name] = {"hist": hist, "params": count_params(model),
                       "macs": count_flops(model), "sec": time.time() - t0}
    print(f"EffNet {name:8s} val_acc={hist['val_acc'][-1]:.2%} | 参数={scale_res[name]['params']:>8,} | MACs={scale_res[name]['macs']/1e6:6.1f}M | {scale_res[name]['sec']:.0f}s", flush=True)

base_acc = scale_res["base"]["hist"]["val_acc"][-1]
print("\n每 +10M MACs 买到的精度（相对 base）:")
for name in ["宽×2", "宽×2+深×2"]:
    d_m = (scale_res[name]["macs"] - scale_res["base"]["macs"]) / 1e6
    d_a = scale_res[name]["hist"]["val_acc"][-1] - base_acc
    print(f"  {name:8s} {d_a:+.2%} / {d_m:.0f}M = {d_a / d_m * 100:+.2f}pt/10M")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
names = list(SCALES)
sc_accs = [scale_res[n]["hist"]["val_acc"][-1] for n in names]
sc_macs = [scale_res[n]["macs"] / 1e6 for n in names]
bars = axes[0].bar(names, sc_accs, color=["#AAAAAA", "#4C72B0", "#DD8452"])
for b, v in zip(bars, sc_accs):
    axes[0].text(b.get_x() + b.get_width() / 2, v, f"{v:.2%}", ha="center", va="bottom", fontsize=9)
axes[0].set_ylim(0.35, 0.62); axes[0].set_ylabel("val_acc")
axes[0].set_title("复合缩放三臂（同 10 epochs）")
axes[1].scatter(sc_macs, sc_accs, s=170, color=["#AAAAAA", "#4C72B0", "#DD8452"], zorder=3)
for n, x, y in zip(names, sc_macs, sc_accs):
    axes[1].annotate(f"{n}\n{x:.0f}M · {y:.1%}", (x, y), textcoords="offset points", xytext=(8, -14), fontsize=8)
axes[1].set_xlabel("MACs (百万/图)"); axes[1].set_ylabel("val_acc")
axes[1].set_title("缩放的成本-收益"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGS / "fig3_scaling.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. 冠军的错误

主实验冠军（MobileNet vs EffNet 中较高者）的混淆矩阵——轻量网的错误分布与重型网（03 ResNet32 的 dog→cat 231）相比是否更"平民"。

In [ ]:
best_name = max(["MobileNet α=1", "EffNet base"], key=lambda n: results[n]["hist"]["val_acc"][-1])
best_acc = results[best_name]["hist"]["val_acc"][-1]
print("主实验冠军:", best_name, f"{best_acc:.2%}")
best_model = results[best_name]["model"]
best_model.eval()
with torch.no_grad():
    pred = best_model(Xte).argmax(1)

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(yte.numpy(), pred.numpy())
fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(CIFAR10_CLASSES, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(10)); ax.set_yticklabels(CIFAR10_CLASSES, fontsize=8)
for i in range(10):
    for j in range(10):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=6)
ax.set_xlabel("预测"); ax.set_ylabel("真实"); ax.set_title(f"{best_name} 混淆矩阵")
plt.colorbar(im)
plt.tight_layout()
plt.savefig(FIGS / "fig4_confusion.png", dpi=150, bbox_inches="tight")
plt.show()

cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
flat = cm_off.ravel().argsort()[::-1][:5]
for k in flat:
    r, c = np.unravel_index(k, cm.shape)
    print(f"  {CIFAR10_CLASSES[r]:10s} → {CIFAR10_CLASSES[c]:10s} : {cm_off[r, c]} 次")

## 8. 总结与下一步

**本项目收获**

1. 深度可分离的账（7.9× 现场 verify）与 MobileNet 的 α 旋钮收益曲线
2. MBConv = 倒残差 + SE——04 的 SEBlock 选学伏笔转正
3. 复合缩放 mini 三臂对照：成本-收益的"每 10M MACs 买多少精度"口径
4. Acc-FLOPs + Acc-params 双图：家族六个架构的性价比地图（DenseNet 两图位置不同 = 参数省/计算贵的可视化证据）

**下一步**：`06_ConvNeXt_vs_ViT`——家族压轴：现代化改造（ConvNeXt 用 Transformer 经验重造 CNN）与跨家族 ViT 同台，为 06 Transformer_Vision 家族铺路。